In [1]:
import pandas as pd

train_data = pd.read_csv("../data/processed/train_data.csv")
validation_data = pd.read_csv("../data/processed/validation_data.csv")
test_data = pd.read_csv("../data/processed/test_data.csv")

#### Select attributes for prediction

In [6]:
# Exclude target, identifiers, and raw timestamps

exclude_columns = [
    "SIZE_CLASS",
    "FIRE_NUMBER",
    "REPORTED_DATE",
    "DISPATCH_DATE",
    "ASSESSMENT_DATETIME"
]

feature_columns = [
    col for col in train_data.columns
    if col not in exclude_columns
]

X_train = train_data[feature_columns].copy()
y_train = train_data["SIZE_CLASS"]

X_validation = validation_data[feature_columns].copy()
y_validation = validation_data["SIZE_CLASS"]

X_test = test_data[feature_columns].copy()
y_test = test_data["SIZE_CLASS"]




#### Convert categorical to Numerical

In [7]:

# Identify categorical columns and numerical columns

for df in [X_train, X_validation, X_test]:
    df["FOREST_AREA"] = df["FOREST_AREA"].astype(str)

numerical_columns = X_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical:", numerical_columns)
print("Categorical:", categorical_columns)

# Encode categorical columns using one-hot encoding
X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    dtype=int
)

X_validation = pd.get_dummies(
    X_validation,
    columns=categorical_columns,
    dtype=int
).reindex(columns=X_train.columns, fill_value=0)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    dtype=int
).reindex(columns=X_train.columns, fill_value=0)


Numerical: ['YEAR', 'ASSESSMENT_HECTARES', 'FIRE_SPREAD_RATE', 'TEMPERATURE', 'RELATIVE_HUMIDITY', 'WIND_SPEED']
Categorical: ['FIRE_ORIGIN', 'DETECTION_AGENT_TYPE', 'DISPATCHED_RESOURCE', 'FIRE_TYPE', 'FIRE_POSITION_ON_SLOPE', 'WEATHER_CONDITIONS_OVER_FIRE', 'WIND_DIRECTION', 'FUEL_TYPE', 'FOREST_AREA']


#### Scale Numerical Attributes

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[numerical_columns] = scaler.fit_transform(
    X_train[numerical_columns]
)

X_validation[numerical_columns] = scaler.transform(
    X_validation[numerical_columns]
)

X_test[numerical_columns] = scaler.transform(
    X_test[numerical_columns]
)

In [9]:
X_train.describe()

,YEAR,ASSESSMENT_HECTARES,FIRE_SPREAD_RATE,TEMPERATURE,RELATIVE_HUMIDITY,WIND_SPEED,FIRE_ORIGIN_DND,FIRE_ORIGIN_Indian Reservation,FIRE_ORIGIN_Metis Settlement,FIRE_ORIGIN_National Park,...,FOREST_AREA_C,FOREST_AREA_E,FOREST_AREA_G,FOREST_AREA_H,FOREST_AREA_L,FOREST_AREA_M,FOREST_AREA_P,FOREST_AREA_R,FOREST_AREA_S,FOREST_AREA_W
count,1.947900e+04,1.947900e+04,1.947900e+04,1.947900e+04,1.947900e+04,1.947900e+04,19479.000000,19479.000000,19479.000000,19479.000000,...,19479.000000,19479.000000,19479.000000,19479.000000,19479.000000,19479.000000,19479.000000,19479.000000,19479.000000,19479.000000
mean,1.520523e-14,3.282964e-18,-3.428873e-17,-6.274108e-17,-7.550816e-17,7.550816e-17,0.004928,0.124544,0.006160,0.000051,...,0.161559,0.082756,0.067663,0.140562,0.126085,0.073207,0.090508,0.072283,0.134401,0.050978
std,1.000026e+00,1.000026e+00,1.000026e+00,1.000026e+00,1.000026e+00,1.000026e+00,0.070031,0.330210,0.078249,0.007165,...,0.368055,0.275520,0.251172,0.347578,0.331953,0.260483,0.286915,0.258963,0.341092,0.219958
min,-1.503854e+00,-4.876325e-02,-7.364747e-01,-7.714571e+00,-2.510845e+00,-1.091786e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-8.133750e-01,-4.876325e-02,-3.311668e-01,-4.068797e-01,-7.051758e-01,-5.909551e-01,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-1.228960e-01,-4.876325e-02,-3.311668e-01,1.344307e-01,-2.537585e-01,-2.153322e-01,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,9.128226e-01,-4.399854e-02,7.414102e-02,6.757411e-01,5.362217e-01,1.602907e-01,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.775921e+00,9.113252e+01,4.019962e+01,2.962778e+00,3.131871e+00,1.017690e+01,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [10]:
X_validation.describe()

,YEAR,ASSESSMENT_HECTARES,FIRE_SPREAD_RATE,TEMPERATURE,RELATIVE_HUMIDITY,WIND_SPEED,FIRE_ORIGIN_DND,FIRE_ORIGIN_Indian Reservation,FIRE_ORIGIN_Metis Settlement,FIRE_ORIGIN_National Park,...,FOREST_AREA_C,FOREST_AREA_E,FOREST_AREA_G,FOREST_AREA_H,FOREST_AREA_L,FOREST_AREA_M,FOREST_AREA_P,FOREST_AREA_R,FOREST_AREA_S,FOREST_AREA_W
count,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000,4174.0,...,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000,4174.000000
mean,-0.005735,0.030685,-0.009154,0.007743,0.041585,-0.002023,0.004312,0.125299,0.004552,0.0,...,0.158840,0.078821,0.068040,0.147101,0.127935,0.078821,0.088884,0.069717,0.132966,0.048874
std,1.001211,1.576448,0.987548,0.992200,1.015546,0.986793,0.065535,0.331098,0.067323,0.0,...,0.365571,0.269492,0.251845,0.354249,0.334057,0.269492,0.284610,0.254700,0.339579,0.215630
min,-1.503854,-0.048763,-0.736475,-6.496622,-2.510845,-1.091786,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-0.813375,-0.048763,-0.331167,-0.406880,-0.648749,-0.590955,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-0.122896,-0.048763,-0.331167,0.134431,-0.253759,-0.215332,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.740203,-0.042745,0.074141,0.675741,0.592649,0.160291,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.775921,75.183276,26.013843,5.682863,3.131871,10.051694,1.000000,1.000000,1.000000,0.0,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [11]:
X_test.describe()

,YEAR,ASSESSMENT_HECTARES,FIRE_SPREAD_RATE,TEMPERATURE,RELATIVE_HUMIDITY,WIND_SPEED,FIRE_ORIGIN_DND,FIRE_ORIGIN_Indian Reservation,FIRE_ORIGIN_Metis Settlement,FIRE_ORIGIN_National Park,...,FOREST_AREA_C,FOREST_AREA_E,FOREST_AREA_G,FOREST_AREA_H,FOREST_AREA_L,FOREST_AREA_M,FOREST_AREA_P,FOREST_AREA_R,FOREST_AREA_S,FOREST_AREA_W
count,4175.000000,4175.000000,4175.000000,4175.000000,4175.000000,4175.000000,4175.000000,4175.000000,4175.000000,4175.000000,...,4175.000000,4175.000000,4175.000000,4175.000000,4175.000000,4175.000000,4175.000000,4175.000000,4175.000000,4175.000000
mean,0.002424,0.048778,0.007418,-0.016598,0.017619,0.002214,0.004551,0.117844,0.006946,0.000240,...,0.151377,0.089102,0.067784,0.148024,0.125749,0.074970,0.092934,0.069940,0.135569,0.044551
std,1.007348,2.700909,1.047373,0.991717,0.988770,1.010546,0.067315,0.322462,0.083063,0.015476,...,0.358459,0.284925,0.251406,0.355166,0.331605,0.263375,0.290375,0.255077,0.342371,0.206340
min,-1.503854,-0.048763,-0.736475,-5.819984,-2.510845,-1.091786,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-0.813375,-0.048763,-0.331167,-0.542207,-0.592322,-0.590955,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-0.122896,-0.048512,-0.331167,0.134431,-0.253759,-0.215332,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.912823,-0.041491,0.074141,0.675741,0.536222,0.160291,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.775921,150.892038,28.040382,2.840983,3.131871,8.298787,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [12]:
X_train.dtypes

YEAR                   float64
ASSESSMENT_HECTARES    float64
FIRE_SPREAD_RATE       float64
TEMPERATURE            float64
RELATIVE_HUMIDITY      float64
                        ...   
FOREST_AREA_M            int64
FOREST_AREA_P            int64
FOREST_AREA_R            int64
FOREST_AREA_S            int64
FOREST_AREA_W            int64
Length: 82, dtype: object

In [13]:
X_validation.dtypes

YEAR                   float64
ASSESSMENT_HECTARES    float64
FIRE_SPREAD_RATE       float64
TEMPERATURE            float64
RELATIVE_HUMIDITY      float64
                        ...   
FOREST_AREA_M            int64
FOREST_AREA_P            int64
FOREST_AREA_R            int64
FOREST_AREA_S            int64
FOREST_AREA_W            int64
Length: 82, dtype: object

In [14]:
X_test.dtypes

YEAR                   float64
ASSESSMENT_HECTARES    float64
FIRE_SPREAD_RATE       float64
TEMPERATURE            float64
RELATIVE_HUMIDITY      float64
                        ...   
FOREST_AREA_M            int64
FOREST_AREA_P            int64
FOREST_AREA_R            int64
FOREST_AREA_S            int64
FOREST_AREA_W            int64
Length: 82, dtype: object